# Decision models as rerankers: the hosted walkthrough

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/treygrainger/ai-powered-search/blob/main/course/decision-models-as-rerankers/hosted_walkthrough.ipynb)

The companion notebook to the talk *Decision Models as Rerankers* (Adam Hevenor, [hevmind.com](https://hevmind.com)). It follows the deck: **Part A** runs the **exact code from the slides** on Jev,
TypeSafe's decision model, for the three request shapes of Part 2 of the deck, on the slides' six-passage example.
**Part B** does the same with [`hev-rerank`](https://github.com/hev/reranker) in one line per shape, then runs five hosted
decision models in every shape on SciFact (Part 3) and writes a custom definition of relevant (Part 4). Headings name
the deck's parts, so you can find your place from a slide.

```python
rerank(query, docs, model="jev" | "perplexity" | "clef" | "clef-flash" | "openai", style="pair" | "batch" | "choice")
```

**What you'll need:** API keys for whichever models you want to run. Set them in Colab's 🔑 *Secrets* panel (with
notebook access on) or as environment variables, or otherwise you'll be prompted to optionally enter them. Part A needs `TYPESAFE_API_KEY` and is skipped without it; in
Part B, models without a key are skipped.

| model | variables |
|---|---|
| `jev` | `TYPESAFE_API_KEY` |
| `perplexity` | `PERPLEXITY_API_KEY` |
| `clef`, `clef-flash` | `CLOUDFLARE_API_TOKEN`, `CLOUDFLARE_ACCOUNT_ID` |
| `openai` | `OPENAI_API_KEY` |

**How long:** about 6 minutes with all five models (most of it Part 3), under a minute with Jev alone. No GPU
needed. No keys at all? [colab_free_tier.ipynb](https://colab.research.google.com/github/treygrainger/ai-powered-search/blob/main/course/decision-models-as-rerankers/colab_free_tier.ipynb)
does the same on an open-weight model on the free Colab GPU.

In [1]:
# Parameters. HEV_RERANK_REF is the git tag or commit of hev-rerank to install.
HEV_RERANK_REF = "v0.3.0"
N_QUERIES = 30      # SciFact test queries per model and shape (pair makes K calls per query)
K = 30              # BM25 candidates per query

In [2]:
# Parameters
HEV_RERANK_REF = "v0.3.0"


In [3]:
# Only needed for Colab. Depenency already added to AIPS notebook environment. Installs typesafe-sdk and hev-rerank.
%pip install -q "hev-rerank @ git+https://github.com/hev/reranker@{HEV_RERANK_REF}" rank_bm25

Note: you may need to restart the kernel to use updated packages.


In [4]:
import os
from getpass import getpass
from IPython.core.magic import register_cell_magic

NEEDS = {"jev": ["TYPESAFE_API_KEY"], "perplexity": ["PERPLEXITY_API_KEY"],
         "clef": ["CLOUDFLARE_API_TOKEN", "CLOUDFLARE_ACCOUNT_ID"], "clef-flash": ["CLOUDFLARE_API_TOKEN", "CLOUDFLARE_ACCOUNT_ID"],
         "openai": ["OPENAI_API_KEY"]}
ALL_VARS = sorted({v for vs in NEEDS.values() for v in vs})

try:                                   # Colab secrets -> environment (values are never printed)
    from google.colab import userdata
    for var in ALL_VARS:
        try:
            os.environ.setdefault(var, userdata.get(var))
        except Exception:
            pass
except ImportError:
    pass

missing = [v for v in ALL_VARS if not os.environ.get(v)]   # prompt only for what's still unset
if missing:
    print("Some keys aren't set. Enter a value, or press Enter to skip.")
    for var in missing:
        try:
            val = getpass(f"{var}: ").strip()
        except Exception:
            val = ""
        if val:
            os.environ[var] = val

MODELS = [m for m, vs in NEEDS.items() if all(os.environ.get(v) for v in vs)]
for m, vs in NEEDS.items():
    print(f"{m:11s} {'ready' if m in MODELS else 'skipped: set ' + ' and '.join(vs)}")
    

@register_cell_magic
def needs_jev(line, cell):
    """%%needs_jev on the first line of a cell: run it only if TYPESAFE_API_KEY is set (Part A calls Jev)."""
    if "jev" in MODELS:
        get_ipython().ex(cell)
    else:
        print("skipped: this cell calls Jev; set TYPESAFE_API_KEY to run it")

jev         ready
perplexity  ready
clef        ready
clef-flash  ready
openai      ready


## Part 1 · History: retrieve, then rerank

BM25 (or any first-stage retriever) is cheap enough to score a whole corpus, so it *retrieves*: it cuts millions of
documents down to a shortlist of 20–100. A reranker is too expensive for the whole corpus, so it only *reorders the
cut*. Two metrics tell those jobs apart:

- **Recall@k**: how many of the relevant documents made it into the top k. For the shortlist as a whole this is fixed
  by the retriever; a reranker can't bring back a document BM25 missed.
- **nDCG@k**: whether the relevant documents are *at the top*, discounted by rank. Reordering the cut is exactly what
  moves it (and recall@5 too, by pulling relevant documents up from places 6–30).

Part 3 measures them on SciFact: BM25's recall over the shortlist (the ceiling), and nDCG@10 before and after
reranking.

# Part A · By hand

The three request shapes from Part 2 of the deck, written out by hand on the deck's six-passage example. No
`hev-rerank` yet: it's imported in Part B.

## Part 2 · The fit: typed questions in, probabilities out

A decision model takes **state** (any text or JSON) plus **typed questions** about it, and returns calibrated
probabilities instead of generated text. The code in Part A is the Python tab of each slide, with the slides' `...`
filled in: all six passages and the full criteria text. Cells that call Jev start with `%%needs_jev`, which skips
them when `TYPESAFE_API_KEY` isn't set.

The deck's example: a query and six search results, keyed `D00`…`D05`. Two passages answer the question, one
for (D04, a trial) and one against (D00, a meta-analysis); relevance is *addresses the query*, not *agrees with it*.
D03 is related background.

In [5]:
query = "does vitamin D supplementation improve bone density in older adults"
docs = {"D00": "Meta-analysis of 23 trials: vitamin D alone did not meaningfully increase bone density in older adults.",
        "D01": "Dietary sources of vitamin D include oily fish, egg yolk and fortified milk. Many older adults fall short.",
        "D02": "Sunscreen reduces vitamin D synthesis in the skin only marginally in practice.",
        "D03": "Vitamin D increases intestinal calcium absorption. Deficiency impairs bone mineralization.",
        "D04": "Randomised trial: daily vitamin D3 with calcium raised hip bone density in adults over 65 versus placebo.",
        "D05": "Astronauts lose 1 to 2 percent of bone density per month in microgravity."}   # query results

part_a = {}   # each shape's scores, kept to compare with hev-rerank in Part B

def show(scores, ranked):
    for d in ranked:
        print(f"  {d}  {scores[d]:5.2f}  {docs[d][:85]}")

## Part 2 · Pairwise: one call per document

Slide *"Typed questions in, probabilities out"*. One request per (query, document) with one **Noul**, a true-or-false
question answered as a probability. The answer is an absolute `p_relevant`: it means the same thing whatever else is
in the list.

In [6]:
%%needs_jev
from typesafe_sdk import TypeSafeClient, Noul

client = TypeSafeClient()
# query, docs: the cell above

def relevance(doc: str) -> float:              # one call per document
    r = client.system_one(
        state={"query": query, "document": doc},
        questions={"relevant": Noul(  # a true-or-false question, answered as a probability
            instructions="`document` is relevant to `query`: "
                         "it contains information that answers or directly addresses it.",
            criteria={"true":  "The document contains information that answers the query "
                               "or directly addresses what it asks about.",
                      "false": "The document is only loosely related, on a similar topic, "
                               "or does not address what the query asks."})},
        model="jev-latest")
    return r.answers["relevant"].noul

p_relevant = {d: relevance(text) for d, text in docs.items()}   # {"D00": 0.98, ...}
ranked = sorted(docs, key=p_relevant.get, reverse=True)

print("pairwise, 6 calls:"); show(p_relevant, ranked); part_a["pair"] = p_relevant

pairwise, 6 calls:
  D00   0.98  Meta-analysis of 23 trials: vitamin D alone did not meaningfully increase bone densit
  D04   0.96  Randomised trial: daily vitamin D3 with calcium raised hip bone density in adults ove
  D03   0.28  Vitamin D increases intestinal calcium absorption. Deficiency impairs bone mineraliza
  D01   0.08  Dietary sources of vitamin D include oily fish, egg yolk and fortified milk. Many old
  D02   0.03  Sunscreen reduces vitamin D synthesis in the skin only marginally in practice.
  D05   0.02  Astronauts lose 1 to 2 percent of bone density per month in microgravity.


## Part 2 · Listwise batch: one call, every document

Slide *"One call, every document"*. All six passages go into one state, keyed `D00`…`D05`, with one Noul per
document that points at its key (`documents.D00`). Still an absolute `p_relevant` per document, from one call.

In [7]:
%%needs_jev
from typesafe_sdk import TypeSafeClient, Noul

# query, docs: above

questions = {
    d: Noul(  # a true-or-false question, answered as a probability
        instructions=f"Document `documents.{d}` is relevant to `query`: "
                     "it contains information that answers or directly addresses it.",
        criteria={"true":  "The document contains information that answers the query "
                           "or directly addresses what it asks about.",
                  "false": "The document is only loosely related, on a similar topic, "
                           "or does not address what the query asks."})
    for d in docs}

r = TypeSafeClient().system_one(state={"query": query, "documents": docs},
                                questions=questions, model="jev-latest")

p_relevant = {d: r.answers[d].noul for d in docs}        # {"D00": 0.97, "D01": 0.06, ...}
ranked = sorted(docs, key=p_relevant.get, reverse=True)

print("listwise batch, 1 call:"); show(p_relevant, ranked); part_a["batch"] = p_relevant

listwise batch, 1 call:
  D00   0.98  Meta-analysis of 23 trials: vitamin D alone did not meaningfully increase bone densit
  D04   0.95  Randomised trial: daily vitamin D3 with calcium raised hip bone density in adults ove
  D03   0.24  Vitamin D increases intestinal calcium absorption. Deficiency impairs bone mineraliza
  D01   0.06  Dietary sources of vitamin D include oily fish, egg yolk and fortified milk. Many old
  D02   0.04  Sunscreen reduces vitamin D synthesis in the skin only marginally in practice.
  D05   0.03  Astronauts lose 1 to 2 percent of bone density per month in microgravity.


## Part 2 · Listwise choice: which is best?

Slide *"One call, one question: which is best?"*. One **Choice** whose options are the documents. The answer is
`p_best`: one unit of probability spread across the list, so it sums to one.

In [8]:
%%needs_jev
from typesafe_sdk import TypeSafeClient, Choice

r = TypeSafeClient().system_one(
    state={"query": query, "documents": docs},
    questions={"best": Choice(  # one question, one option per document
        instructions="Which document in `documents` best answers or most directly addresses `query`?",
        criteria={d: f"documents.{d}" for d in docs})},
    model="jev-latest")

p_best = r.answers["best"].probabilities   # {"D00": 0.85, "D04": 0.15, "D01": 0.0, ...}
ranked = sorted(docs, key=p_best.get, reverse=True)

print(f"listwise choice, 1 call (p_best sums to {sum(p_best.values()):.2f}):"); show(p_best, ranked); part_a["choice"] = p_best

listwise choice, 1 call (p_best sums to 1.00):
  D00   0.81  Meta-analysis of 23 trials: vitamin D alone did not meaningfully increase bone densit
  D04   0.19  Randomised trial: daily vitamin D3 with calcium raised hip bone density in adults ove
  D01   0.00  Dietary sources of vitamin D include oily fish, egg yolk and fortified milk. Many old
  D02   0.00  Sunscreen reduces vitamin D synthesis in the skin only marginally in practice.
  D03   0.00  Vitamin D increases intestinal calcium absorption. Deficiency impairs bone mineraliza
  D05   0.00  Astronauts lose 1 to 2 percent of bone density per month in microgravity.


## Part 2 · When to use which

| shape | calls for N docs | score | threshold? | use it when |
|---|---|---|---|---|
| **pairwise** | N | `p_relevant`, absolute | yes | you want the most robust per-document judgement and calls are cheap or parallel |
| **listwise batch** | 1 (or a few) | `p_relevant`, absolute | yes | you want pairwise's absolute scores at a fraction of the calls; the model also sees the other candidates |
| **listwise choice** | 1, always | `p_best`, sums to one | **no** | you need the single best answer, or few documents are relevant |

Pairwise and batch say *how likely is this one relevant*, so several documents can be near 1 (D00 and D04 here) and a
threshold prunes the rest. Choice says *which one wins*: D00 and D04 split one unit between them, and a document's
`p_best` depends on what else is in the list, so a fixed cut-off means nothing. Which shape ranks best depends on the
model: benchmark the shape on yours (Part 3).

# Part B · The same three shapes with hev-rerank

[`hev-rerank`](https://github.com/hev/reranker) wraps everything in Part A behind one call:
`rerank(query, docs, model=..., style=...)`, returning `RerankResult(index, score, document)` best-first. `style=` is
the shape: `"pair"`, `"batch"` or `"choice"`.

`prompt=` sets the wording. `slides` below is Part A's wording, so Part B asks exactly Part A's questions. (The
library's own default wording differs slightly per model; Part 3 uses the default.)

In [9]:
import hev_rerank
from hev_rerank import rerank, MODELS as TABLE
print(f"hev-rerank {hev_rerank.__version__}")

slides = {
    "pair_question": "`document` is relevant to `query`: it contains information that answers or directly addresses it.",
    "question": "Document `documents.{id}` is relevant to `query`: it contains information that answers or directly addresses it.",
    "choice_question": "Which document in `documents` best answers or most directly addresses `query`?",
    "criteria": {"true": "The document contains information that answers the query or directly addresses what it asks about.",
                 "false": "The document is only loosely related, on a similar topic, or does not address what the query asks."},
}
ids, texts = list(docs), list(docs.values())    # rerank takes a list; its result's .index is the position in it

def compare(style, hits):
    """Part B's scores and order next to Part A's for the same shape."""
    b = {ids[h.index]: h.score for h in hits}
    a = part_a[style]
    order_a, order_b = sorted(ids, key=a.get, reverse=True), [ids[h.index] for h in hits]
    print(f"{style}:  doc   Part A  Part B")
    for d in order_b:
        print(f"         {d}  {a[d]:6.3f}  {b[d]:6.3f}")
    print(f"  Part A order {order_a}\n  Part B order {order_b}   same order: {order_a == order_b}, "
          f"max |score difference| {max(abs(a[d] - b[d]) for d in ids):.3f}\n")

hev-rerank 0.3.0


In [10]:
%%needs_jev
for style in ["pair", "batch", "choice"]:
    hits = rerank(query, texts, model="jev", style=style, prompt=slides)
    compare(style, hits)

pair:  doc   Part A  Part B
         D00   0.980   0.980
         D04   0.960   0.960
         D03   0.280   0.290
         D01   0.080   0.080
         D02   0.030   0.030
         D05   0.020   0.020
  Part A order ['D00', 'D04', 'D03', 'D01', 'D02', 'D05']
  Part B order ['D00', 'D04', 'D03', 'D01', 'D02', 'D05']   same order: True, max |score difference| 0.010

batch:  doc   Part A  Part B
         D00   0.980   0.970
         D04   0.950   0.930
         D03   0.240   0.240
         D01   0.060   0.060
         D02   0.040   0.040
         D05   0.030   0.030
  Part A order ['D00', 'D04', 'D03', 'D01', 'D02', 'D05']
  Part B order ['D00', 'D04', 'D03', 'D01', 'D02', 'D05']   same order: True, max |score difference| 0.020

choice:  doc   Part A  Part B
         D00   0.810   0.840
         D04   0.190   0.160
         D01   0.000   0.000
         D02   0.000   0.000
         D03   0.000   0.000
         D05   0.000   0.000
  Part A order ['D00', 'D04', 'D01', 'D02', 'D03', 'D05']
 

Each `rerank(...)` call is one shape from Part A, with the same questions to the same model. Scores can move by a
point or two between calls: that's the hosted model's own run-to-run variation, not the library.

### Every model, every shape

`model=` swaps the decision model; each one gets its own default wording from here on (the one it was benchmarked
with). Starred: the shape `style="auto"` picks for that model, its best measured on the benchmark.

In [11]:
for m in MODELS:
    for style in ["pair", "batch", "choice"]:
        hits = rerank(query, texts, model=m, style=style)
        star = "*" if TABLE[m][2] == style else " "
        print(f"{m:11s} {style:6s}{star} " + "  ".join(f"{ids[h.index]} {h.score:.2f}" for h in hits))
if not MODELS:
    print("No keys set: nothing to call. Set a key above, or use colab_free_tier.ipynb.")

jev         pair    D00 0.98  D04 0.95  D03 0.28  D01 0.07  D02 0.03  D05 0.03
jev         batch * D00 0.98  D04 0.94  D03 0.25  D01 0.06  D02 0.04  D05 0.03
jev         choice  D00 0.86  D04 0.14  D01 0.00  D02 0.00  D03 0.00  D05 0.00
perplexity  pair    D00 0.99  D04 0.96  D03 0.17  D01 0.02  D05 0.01  D02 0.01
perplexity  batch   D04 1.00  D00 0.99  D03 0.19  D01 0.03  D05 0.00  D02 0.00
perplexity  choice* D00 0.79  D04 0.20  D03 0.00  D01 0.00  D05 0.00  D02 0.00
clef        pair  * D04 0.97  D00 0.97  D03 0.37  D01 0.06  D05 0.03  D02 0.03
clef        batch   D04 0.94  D00 0.91  D03 0.39  D05 0.04  D02 0.03  D01 0.03
clef        choice  D00 0.48  D04 0.47  D03 0.02  D01 0.01  D02 0.01  D05 0.01
clef-flash  pair  * D04 0.88  D00 0.86  D03 0.21  D01 0.09  D02 0.08  D05 0.07
clef-flash  batch   D04 0.93  D00 0.88  D03 0.39  D01 0.07  D05 0.05  D02 0.04
clef-flash  choice  D00 0.64  D04 0.21  D03 0.06  D01 0.04  D02 0.03  D05 0.03
openai      pair    D00 1.00  D04 1.00  D03 0.01  D0

## Part 3 · Benchmark: SciFact, BM25 then every model in every shape

SciFact (from BEIR) has 5,183 paper abstracts and 300 test claims with human relevance labels. BM25 retrieves the top
K for each claim; each model and shape reranks them; nDCG@10 and latency are measured.

In [12]:
import csv, io, json, math, random, re, urllib.request, zipfile
from rank_bm25 import BM25Okapi

# SciFact from BEIR (5,183 abstracts, 300 test queries with human relevance labels), 2.8 MB.
raw = urllib.request.urlopen("https://public.ukp.informatik.tu-darmstadt.de/thakur/BEIR/datasets/scifact.zip").read()
z = zipfile.ZipFile(io.BytesIO(raw))
corpus = {d["_id"]: {"title": d["title"], "text": d["text"]} for d in map(json.loads, z.open("scifact/corpus.jsonl"))}
queries = {q["_id"]: q["text"] for q in map(json.loads, z.open("scifact/queries.jsonl"))}
qrels = {}
for row in csv.DictReader(io.TextIOWrapper(z.open("scifact/qrels/test.tsv")), delimiter="\t"):
    qrels.setdefault(row["query-id"], {})[row["corpus-id"]] = int(row["score"])

tok = lambda s: re.findall(r"\w+", s.lower())
corpus_ids = list(corpus)
bm25 = BM25Okapi([tok(corpus[i]["title"] + " " + corpus[i]["text"]) for i in corpus_ids])

def bm25_top(q, k=K):
    scores = bm25.get_scores(tok(q))
    return [corpus_ids[i] for i in sorted(range(len(corpus_ids)), key=lambda i: -scores[i])[:k]]

def ndcg10(ranked, rel):
    dcg = sum(rel.get(d, 0) / math.log2(i + 2) for i, d in enumerate(ranked[:10]))
    ideal = sum(r / math.log2(i + 2) for i, r in enumerate(sorted(rel.values(), reverse=True)[:10]))
    return dcg / ideal if ideal else 0.0

qids = sorted(qrels)[:N_QUERIES]          # the first N test queries, sorted by id: the same subset every run
shortlists = {q: bm25_top(queries[q]) for q in qids}
bm25_ndcg = sum(ndcg10(shortlists[q], qrels[q]) for q in qids) / len(qids)
recall = sum(len(set(shortlists[q]) & set(qrels[q])) / len(qrels[q]) for q in qids) / len(qids)
print(f"{len(qids)} SciFact test queries, BM25 top-{K}: nDCG@10 {bm25_ndcg:.3f}, recall@{K} {recall:.3f} "
      "(a reranker can only reorder what BM25 found)")

30 SciFact test queries, BM25 top-30: nDCG@10 0.695, recall@30 0.867 (a reranker can only reorder what BM25 found)


In [13]:
import time
from concurrent.futures import ThreadPoolExecutor
from hev_rerank import ProviderError

def pct(xs, p):
    xs = sorted(xs)
    return xs[min(len(xs) - 1, int(round(p / 100 * (len(xs) - 1))))]

def run(model, style):
    def one(q):
        t = time.perf_counter()
        hits = rerank(queries[q], [corpus[d] for d in shortlists[q]], model=model, style=style)
        return q, hits, time.perf_counter() - t
    # Four queries in flight for batch and choice; pair already sends 8 calls at a time per query, so one query
    # at a time there keeps under the providers' rate limits.
    with ThreadPoolExecutor(1 if style == "pair" else 4) as ex:
        return list(ex.map(one, qids))

results, scores = {}, {}
for m in MODELS:
    for style in ["pair", "batch", "choice"]:
        try:
            out = run(m, style)
        except ProviderError as e:                 # e.g. no beta access on this key: skip, keep going
            print(f"{m:11s} {style:6s} skipped: {str(e)[:120]}")
            continue
        nd = sum(ndcg10([shortlists[q][h.index] for h in hits], qrels[q]) for q, hits, _ in out) / len(out)
        lat = [s for _, _, s in out]
        results[m, style] = {"ndcg": nd, "p50": pct(lat, 50), "p95": pct(lat, 95)}
        scores[m, style] = {q: {shortlists[q][h.index]: h.score for h in hits} for q, hits, _ in out}
        print(f"{m:11s} {style:6s} nDCG@10 {bm25_ndcg:.3f} -> {nd:.3f}   p50 {pct(lat, 50):.2f} s  p95 {pct(lat, 95):.2f} s per query")

jev         pair   nDCG@10 0.695 -> 0.806   p50 0.50 s  p95 0.58 s per query
jev         batch  nDCG@10 0.695 -> 0.795   p50 0.39 s  p95 0.57 s per query
jev         choice nDCG@10 0.695 -> 0.800   p50 0.46 s  p95 0.71 s per query
perplexity  pair   nDCG@10 0.695 -> 0.792   p50 4.27 s  p95 4.79 s per query
perplexity  batch  nDCG@10 0.695 -> 0.798   p50 0.99 s  p95 2.62 s per query
perplexity  choice nDCG@10 0.695 -> 0.779   p50 0.88 s  p95 1.41 s per query
clef        pair   nDCG@10 0.695 -> 0.776   p50 1.34 s  p95 1.59 s per query
clef        batch  nDCG@10 0.695 -> 0.751   p50 1.87 s  p95 2.57 s per query
clef        choice nDCG@10 0.695 -> 0.815   p50 1.44 s  p95 1.95 s per query
clef-flash  pair   nDCG@10 0.695 -> 0.782   p50 1.08 s  p95 2.05 s per query
clef-flash  batch  nDCG@10 0.695 -> 0.653   p50 1.08 s  p95 3.48 s per query
clef-flash  choice nDCG@10 0.695 -> 0.759   p50 0.91 s  p95 1.74 s per query
openai      pair   nDCG@10 0.695 -> 0.774   p50 1.06 s  p95 1.32 s per query

In [14]:
from IPython.display import Markdown
rows = [f"| {m} | {s} | {r['ndcg']:.3f} | {r['ndcg'] - bm25_ndcg:+.3f} | {r['p50']:.2f} | {r['p95']:.2f} |"
        for (m, s), r in results.items()]
Markdown(f"{len(qids)} SciFact test queries, BM25 top-{K} (nDCG@10 {bm25_ndcg:.3f}). Latency per query, from wherever this ran.\n\n"
         "| model | shape | nDCG@10 | Δ vs BM25 | p50 s | p95 s |\n|---|---|---|---|---|---|\n" + "\n".join(rows))

30 SciFact test queries, BM25 top-30 (nDCG@10 0.695). Latency per query, from wherever this ran.

| model | shape | nDCG@10 | Δ vs BM25 | p50 s | p95 s |
|---|---|---|---|---|---|
| jev | pair | 0.806 | +0.112 | 0.50 | 0.58 |
| jev | batch | 0.795 | +0.100 | 0.39 | 0.57 |
| jev | choice | 0.800 | +0.106 | 0.46 | 0.71 |
| perplexity | pair | 0.792 | +0.098 | 4.27 | 4.79 |
| perplexity | batch | 0.798 | +0.103 | 0.99 | 2.62 |
| perplexity | choice | 0.779 | +0.084 | 0.88 | 1.41 |
| clef | pair | 0.776 | +0.081 | 1.34 | 1.59 |
| clef | batch | 0.751 | +0.057 | 1.87 | 2.57 |
| clef | choice | 0.815 | +0.121 | 1.44 | 1.95 |
| clef-flash | pair | 0.782 | +0.088 | 1.08 | 2.05 |
| clef-flash | batch | 0.653 | -0.041 | 1.08 | 3.48 |
| clef-flash | choice | 0.759 | +0.064 | 0.91 | 1.74 |
| openai | pair | 0.774 | +0.079 | 1.06 | 1.32 |
| openai | batch | 0.832 | +0.138 | 0.67 | 1.15 |
| openai | choice | 0.805 | +0.110 | 0.52 | 0.93 |

With thirty queries, differences of a few points are within noise; the
[benchmark](https://hevmind.com/research/rerankers) uses 923 queries over three corpora and is where
`style="auto"` comes from.

### Calibration: do batch probabilities mean what they say?

A batch (or pair) score is a probability that the document is relevant. If it is calibrated, documents scored
around 0.8 should be relevant about 80% of the time. Bucket every (query, candidate) score and compare with the
human labels. (SciFact labels are sparse, about one relevant abstract per query, so the top buckets read low:
some "irrelevant" documents with high scores are unlabeled near-duplicates.)

In [15]:
for m in MODELS:
    if (m, "batch") not in scores:
        continue
    pairs = [(s, d in qrels[q]) for q, ds in scores[m, "batch"].items() for d, s in ds.items()]
    print(f"\n{m}, batch: {len(pairs)} scored candidates")
    print("  score bucket   n     relevant")
    for lo, hi in [(0, .01), (.01, .1), (.1, .5), (.5, .9), (.9, 1.01)]:
        b = [r for s, r in pairs if lo <= s < hi]
        if b:
            print(f"  [{lo:.2f}, {min(hi, 1):.2f})  {len(b):5d}   {sum(b) / len(b):6.1%}")


jev, batch: 900 scored candidates
  score bucket   n     relevant
  [0.01, 0.10)    766     0.0%
  [0.10, 0.50)     98     9.2%
  [0.50, 0.90)     18    38.9%
  [0.90, 1.00)     18    77.8%

perplexity, batch: 900 scored candidates
  score bucket   n     relevant
  [0.00, 0.01)    666     0.0%
  [0.01, 0.10)    153     0.7%
  [0.10, 0.50)     45    17.8%
  [0.50, 0.90)     18    33.3%
  [0.90, 1.00)     18    83.3%

clef, batch: 900 scored candidates
  score bucket   n     relevant
  [0.01, 0.10)    445     0.0%
  [0.10, 0.50)    412     1.2%
  [0.50, 0.90)     33    48.5%
  [0.90, 1.00)     10    90.0%

clef-flash, batch: 900 scored candidates
  score bucket   n     relevant
  [0.01, 0.10)     59     1.7%
  [0.10, 0.50)    754     0.8%
  [0.50, 0.90)     83    22.9%
  [0.90, 1.00)      4   100.0%

openai, batch: 900 scored candidates
  score bucket   n     relevant
  [0.00, 0.01)    805     0.1%
  [0.01, 0.10)     43     2.3%
  [0.10, 0.50)     17     5.9%
  [0.50, 0.90)      8    62

## Choice can't be thresholded

Pair and batch scores are absolute, so they sort *and* prune: `threshold=0.1` drops candidates the model is
confident are irrelevant. Choice probabilities sum to one across the list; a document's choice score depends on
what else is in the list, so a fixed cut-off means nothing. `hev-rerank` raises instead of guessing, including
for `style="auto"` on models whose best shape is choice (Perplexity, OpenAI). This needs no key: it fails before
any call.

In [16]:
try:
    rerank(query, texts, model="perplexity", style="choice", threshold=0.1)
except ValueError as e:
    print("ValueError:", e)
for m in MODELS:
    style = "batch" if TABLE[m][2] == "choice" else "auto"
    kept = rerank(query, texts, model=m, style=style, threshold=0.1)
    print(f"{m} {style}: keeps {[ids[h.index] for h in kept]} of {len(docs)} at threshold 0.1")

ValueError: threshold can't be used with style='choice': choice probabilities sum to one across the list, so they rank documents but can't be thresholded. Use style='pair' or 'batch'.
jev auto: keeps ['D00', 'D04', 'D03'] of 6 at threshold 0.1
perplexity batch: keeps ['D04', 'D00', 'D03'] of 6 at threshold 0.1
clef auto: keeps ['D04', 'D00', 'D03'] of 6 at threshold 0.1
clef-flash auto: keeps ['D04', 'D00', 'D03'] of 6 at threshold 0.1
openai batch: keeps ['D00', 'D04', 'D03'] of 6 at threshold 0.1


## Part 4 · Customize: your own definition of relevant

The question and its true/false criteria are plain text: change them to say what *relevant* means for your corpus.
`prompt=` overrides any of `question` (batch, with `{id}` for the document id), `pair_question`, `choice_question` and
`criteria`; anything you leave out keeps the default. Here, a stricter notion: only **clinical trial evidence**
counts, so the mechanism passage (D03) drops while the trial (D04) and the meta-analysis (D00) stay.

The deck also covers **intent**: what the person searching is trying to get done, which can change what counts as
relevant for the same query (a clinician choosing a treatment, a student writing a review). It's a concept, not a
`hev-rerank` parameter; the place to express it is the same as here, the wording of the question and criteria.

In [17]:
strict = {
    "question": "Does document `documents.{id}` report trial or meta-analysis evidence on the question in `query`?",
    "pair_question": "Does the document report trial or meta-analysis evidence on the question in the query?",
    "criteria": {"true": "The document reports results of a trial or a meta-analysis on the query's question, whatever the result.",
                 "false": "Background, mechanisms, dietary advice or other populations: not trial evidence on the question."},
}

for m in MODELS:
    style = "batch" if TABLE[m][2] == "choice" else TABLE[m][2]
    default = {ids[h.index]: h.score for h in rerank(query, texts, model=m, style=style)}
    custom = {ids[h.index]: h.score for h in rerank(query, texts, model=m, style=style, prompt=strict)}
    print(f"\n{m} ({style})   default -> trial-evidence prompt")
    for d in sorted(custom, key=custom.get, reverse=True):
        print(f"  {d}  {default[d]:6.3f} -> {custom[d]:6.3f}  {docs[d][:70]}")


jev (batch)   default -> trial-evidence prompt
  D00   0.970 ->  0.990  Meta-analysis of 23 trials: vitamin D alone did not meaningfully incre
  D04   0.940 ->  0.970  Randomised trial: daily vitamin D3 with calcium raised hip bone densit
  D02   0.040 ->  0.020  Sunscreen reduces vitamin D synthesis in the skin only marginally in p
  D03   0.240 ->  0.020  Vitamin D increases intestinal calcium absorption. Deficiency impairs 
  D05   0.030 ->  0.020  Astronauts lose 1 to 2 percent of bone density per month in microgravi
  D01   0.060 ->  0.010  Dietary sources of vitamin D include oily fish, egg yolk and fortified

perplexity (batch)   default -> trial-evidence prompt
  D04   1.000 ->  1.000  Randomised trial: daily vitamin D3 with calcium raised hip bone densit
  D00   0.994 ->  0.999  Meta-analysis of 23 trials: vitamin D alone did not meaningfully incre
  D05   0.004 ->  0.002  Astronauts lose 1 to 2 percent of bone density per month in microgravi
  D01   0.026 ->  0.001  Dietary 

## Part 5 · Conclusions

- A decision model is a reranker once you ask it *"is this document relevant to the query?"* as a typed question:
  state in, calibrated probabilities out.
- Three request shapes: **pairwise** (one call per document), **listwise batch** (one call, one Noul per document) and
  **listwise choice** (one call, one Choice). Pairwise and batch give absolute probabilities you can threshold; choice
  gives relative ones you can only sort.
- The best shape depends on the model, so benchmark it on yours: with `hev-rerank` that's one argument.
- *Relevant* is a prompt. Write your own definition.

More: the benchmark at [hevmind.com/research/rerankers](https://hevmind.com/research/rerankers), code at
[hev/rerank-bench](https://github.com/hev/rerank-bench), and this library at
[hev/reranker](https://github.com/hev/reranker).